# Modelo de Classificação com XGBoost + Optuna
Notebook gerado automaticamente para treinar um modelo usando os dados da planilha **BASE DE DADOS PEDE 2024**.

Etapas:
1. Carregamento dos dados
2. Análise exploratória básica
3. Pré-processamento
4. Otimização de hiperparâmetros com Optuna
5. Treinamento do modelo XGBClassifier
6. Avaliação com métricas de classificação

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from xgboost import XGBClassifier
import optuna

In [16]:
# Carregar dados
path = "../files/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
df = pd.read_excel(path, sheet_name="PEDE2024")

df.columns = (
    df.columns.str.strip()        # remove espaços extras
             .str.lower()         # tudo em minúsculo
             .str.replace(" ", "_")  # substitui espaço por underscore
)

print("Shape:", df.shape)
df.head()

Shape: (1156, 50)


,ra,fase,inde_2024,pedra_2024,turma,nome_anonimizado,data_de_nasc,idade,gênero,ano_ingresso,...,ipv,ian,fase_ideal,defasagem,destaque_ieg,destaque_ida,destaque_ipv,escola,ativo/_inativo,ativo/_inativo.1
0,RA-1275,ALFA,7.611367,Ametista,ALFA A - G0/G1,Aluno-1275,2016-07-28,8,Masculino,2024,...,5.446667,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,EE Chácara Florida II,Cursando,Cursando
1,RA-1276,ALFA,8.002867,Topázio,ALFA A - G0/G1,Aluno-1276,2016-10-16,8,Feminino,2024,...,7.050000,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,EE Chácara Florida II,Cursando,Cursando
2,RA-1277,ALFA,7.9522,Ametista,ALFA A - G0/G1,Aluno-1277,2016-08-16,8,Masculino,2024,...,7.046667,10.0,ALFA (1° e 2° ano),0,NaN,NaN,NaN,EE Dom Pedro Villas Boas de Souza,Cursando,Cursando
3,RA-868,ALFA,7.156367,Ametista,ALFA A - G0/G1,Aluno-868,2015-11-08,8,Masculino,2023,...,7.213333,5.0,Fase 1 (3° e 4° ano),-1,NaN,NaN,NaN,EE Chácara Florida II,Cursando,Cursando
4,RA-1278,ALFA,5.4442,Quartzo,ALFA A - G0/G1,Aluno-1278,2015-03-22,9,Masculino,2024,...,4.173333,5.0,Fase 1 (3° e 4° ano),-1,NaN,NaN,NaN,EM Etelvina Delfim Simões,Cursando,Cursando


In [17]:
numeric_cols = df.select_dtypes(include='number').columns
categorical_cols = df.select_dtypes(exclude='number').columns

numeric_cols, categorical_cols

(Index(['idade', 'ano_ingresso', 'inde_22', 'inde_23', 'cg', 'cf', 'ct',
        'nº_av', 'rec_av1', 'rec_av2', 'iaa', 'ieg', 'ips', 'ipp',
        'rec_psicologia', 'ida', 'mat', 'por', 'ing', 'indicado', 'atingiu_pv',
        'ipv', 'ian', 'defasagem', 'destaque_ieg', 'destaque_ida',
        'destaque_ipv'],
       dtype='object'),
 Index(['ra', 'fase', 'inde_2024', 'pedra_2024', 'turma', 'nome_anonimizado',
        'data_de_nasc', 'gênero', 'instituição_de_ensino', 'pedra_20',
        'pedra_21', 'pedra_22', 'pedra_23', 'avaliador1', 'avaliador2',
        'avaliador3', 'avaliador4', 'avaliador5', 'avaliador6', 'fase_ideal',
        'escola', 'ativo/_inativo', 'ativo/_inativo.1'],
       dtype='object'))

In [18]:
for col in categorical_cols:
    print(f'\n{col}')
    display(df[col].value_counts(normalize=True).head(5))


ra


ra
RA-1275    0.000865
RA-1276    0.000865
RA-1277    0.000865
RA-868     0.000865
RA-1278    0.000865
Name: proportion, dtype: float64


fase


fase
ALFA    0.169550
9       0.032872
7E      0.021626
8E      0.019896
4M      0.015571
Name: proportion, dtype: float64


inde_2024


...

escola
EE Donizetti Aparecido Leite Professor    0.094372
EE Maria André Schunck Dona               0.086580
EE Dom Pedro Villas Boas de Souza         0.073593
Colégio Arco Íris Evolução                0.073593
EE Helio Luiz Dobrochinski Prof           0.061472
Name: proportion, dtype: float64


ativo/_inativo


ativo/_inativo
Cursando    1.0
Name: proportion, dtype: float64


ativo/_inativo.1


ativo/_inativo.1
Cursando    1.0
Name: proportion, dtype: float64

In [19]:
df['defasagem'].value_counts(normalize=True)

defasagem
 0    0.419550
-1    0.381488
 1    0.102941
-2    0.077855
 2    0.013841
-3    0.002595
 3    0.001730
Name: proportion, dtype: float64

In [20]:
cols = ['ipv', 'ips', 'iaa', 'ieg', 'nº_av', 'ida', 'defasagem']
df = df[cols]

target = "defasagem"

# Criar mapeamento das classes para índices não-negativos
unique_classes = sorted(df[target].dropna().unique())
class_mapping = {c: i for i, c in enumerate(unique_classes)}

# Aplicar mapeamento
y = df[target].map(class_mapping)
X = df.drop(columns=[target])
print(X.shape, y.shape)

(1156, 6) (1156,)


In [21]:
# Transformar variáveis categóricas em numéricas
# X = pd.get_dummies(X, drop_first=True)

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "use_label_encoder": False,
        "eval_metric": "mlogloss"
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average="weighted")

In [23]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Melhores hiperparâmetros:", study.best_params)

[I 2026-03-08 11:52:59,716] A new study created in memory with name: no-name-10fca90e-da7f-4f89-8bad-4c51922ecd12
D:\Repositories\PythonProjects\tc_datathon\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:53:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-03-08 11:53:01,275] Trial 0 finished with value: 0.4687354312354312 and parameters: {'n_estimators': 837, 'max_depth': 3, 'learning_rate': 0.03722206902784174, 'subsample': 0.7703254206906018, 'colsample_bytree': 0.6727168329602344, 'gamma': 0.06188515305409814, 'reg_alpha': 0.8514043341773547, 'reg_lambda': 1.2852970280954468}. Best is trial 0 with value: 0.4687354312354312.
D:\Repositories\PythonProjects\tc_datathon\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:53:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder

Melhores hiperparâmetros: {'n_estimators': 334, 'max_depth': 9, 'learning_rate': 0.2069156090114873, 'subsample': 0.9365901741809173, 'colsample_bytree': 0.779866385763346, 'gamma': 4.356565632693918, 'reg_alpha': 0.04318841768441173, 'reg_lambda': 0.7530403243410115}


In [24]:
# 8. Treinar modelo final
best_model = XGBClassifier(**study.best_params)
best_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.779866385763346
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [25]:
# 9. Avaliação
y_pred = best_model.predict(X_test)

print("Classification Report:\n", classification_report(y_test, y_pred, labels=np.unique(y_test), zero_division=0))

print("Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))

# ROC AUC multi-class
y_proba = best_model.predict_proba(X_test)
print("ROC AUC (ovo):", roc_auc_score(y_test, y_proba, average="weighted"))

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00        18
           2       0.52      0.62      0.56        89
           3       0.48      0.62      0.54        97
           4       0.00      0.00      0.00        24
           5       0.00      0.00      0.00         3

    accuracy                           0.50       232
   macro avg       0.17      0.21      0.18       232
weighted avg       0.40      0.50      0.44       232

Precision: 0.3997381262199089
Recall: 0.4956896551724138
F1 Score: 0.44240327861017514


ValueError: multi_class must be in ('ovo', 'ovr')